In [1]:
import cv2
import numpy as np


import open3d as o3d
import numpy as np



Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
#manually align 1st frames
cap1 = cv2.VideoCapture('cam1.mp4')
cap2 = cv2.VideoCapture('cam2.mp4')

orb = cv2.ORB_create()

while True:
    ret1, frame1 = cap1.read()
    ret2, frame2 = cap2.read()

    if not ret1 or not ret2:
        break

        
    h, w = frame1.shape[:2]

    dx, dy = 0, 0

    while True:
        canvas = frame1.copy()

        # overlay frame2 with offset
        x1 = max(0, dx)
        y1 = max(0, dy)

        x2 = max(0, -dx)
        y2 = max(0, -dy)

        w_crop = min(w - x1, frame2.shape[1] - x2)
        h_crop = min(h - y1, frame2.shape[0] - y2)

        if w_crop > 0 and h_crop > 0:
            overlay = frame2[y2:y2+h_crop, x2:x2+w_crop]
            canvas[y1:y1+h_crop, x1:x1+w_crop] = cv2.addWeighted(
                canvas[y1:y1+h_crop, x1:x1+w_crop], 0.5,
                overlay, 0.5, 0
            )

        cv2.imshow("Align frames", canvas)

        key = cv2.waitKey(0)

        if key == ord('a'): dx -= 1
        elif key == ord('d'): dx += 1
        elif key == ord('w'): dy -= 1
        elif key == ord('s'): dy += 1
        elif key == 27:  # ESC
            break

    print("Final offset:")
    print("dx =", dx)
    print("dy =", dy)
    break

    # if cv2.waitKey(1) == 27:
    #     break

cap1.release()
cap2.release()
cv2.destroyAllWindows()

Final offset:
dx = 344
dy = 0


In [ ]:
#manually align 1st frames
cap1 = cv2.VideoCapture('cam1.mp4')
cap2 = cv2.VideoCapture('cam2.mp4')

orb = cv2.ORB_create()

while True:
    ret1, frame1 = cap1.read()
    ret2, frame2 = cap2.read()
    h, w = frame2.shape[:2]

    M = np.float32([
        [1, 0, dx],
        [0, 1, dy]
    ])

    frame2_trans = cv2.warpAffine(frame2, M, (w, h))
        
    combined = cv2.hconcat([frame1, frame2_trans])

    cv2.imshow("Frames", combined)
    cv2.waitKey(0)
cv2.destroyAllWindows()


In [ ]:
dx = 344
dy = 0
# ---- Read camera frames ----
cap1 = cv2.VideoCapture('cam1.mp4')
cap2 = cv2.VideoCapture('cam2.mp4')

orb = cv2.ORB_create()

while True:
    ret1, frame1 = cap1.read()
    ret2, frame2 = cap2.read()

    if not ret1 or not ret2:
        break
    h, w = frame2.shape[:2]

    M = np.float32([
        [1, 0, dx],
        [0, 1, dy]
    ])

    frame2_trans = cv2.warpAffine(frame2, M, (w, h))

    gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(frame2_trans, cv2.COLOR_BGR2GRAY)

    # Detect features
    kp1, des1 = orb.detectAndCompute(gray1, None)
    kp2, des2 = orb.detectAndCompute(gray2, None)

    # Match features
    # bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    # matches = bf.match(des1, des2)

    bf = cv2.BFMatcher()
    matches = bf.match(des1, des2)
    threshold = 150  # adjust depending on data
    matches = [m for m in matches if m.distance < threshold]
    # matches = bf.knnMatch(des1, des2, k=2)

    # good_matches = []
    # for m, n in matches:
    #     if m.distance < 0.75 * n.distance:
    #         good_matches.append(m)

    # good_matches = sorted(good_matches, key=lambda x: x.distance)

    # good_matches = good_matches[:50]
    match_img = cv2.drawMatches(
    gray1, kp1,
    gray2, kp2,
    matches[:500], None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
    )

    cv2.imshow("Feature Matches", match_img)
    pts1 = []
    pts2 = []

    for m in matches[:50]:
        pts1.append(kp1[m.queryIdx].pt)
        pts2.append(kp2[m.trainIdx].pt)

    pts1 = np.array(pts1).T
    pts2 = np.array(pts2).T
    img1_kp = cv2.drawKeypoints(
    gray1, kp1, None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
    )

    img2_kp = cv2.drawKeypoints(
        gray2, kp2, None,
        flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
    )

    cv2.imshow("Frame 1 Keypoints", img1_kp)
    cv2.imshow("Frame 2 Keypoints", img2_kp)

    # # ---- Triangulation ----
    # points_4d = cv2.triangulatePoints(P1, P2, pts1, pts2)

    # # Convert homogeneous → 3D
    # points_3d = points_4d[:3] / points_4d[3]
    
    # points = points_3d.T

    # # pcd = o3d.geometry.PointCloud()
    # # pcd.points = o3d.utility.Vector3dVector(points)

    # #o3d.visualization.draw_geometries([pcd])

    # print("3D Points:\n", points_3d.T[:5])

    # cv2.imshow("cam1", frame1)
    # cv2.imshow("cam2", frame2)
    while True:
        key = cv2.waitKey(0) & 0xFF
        if key == ord('n'):
            break
        elif key == ord('q'):
            cap1.release()
            cap2.release()
            cv2.destroyAllWindows()
            exit()

    # if cv2.waitKey(1) == 27:
    #     break

cap1.release()
cap2.release()
cv2.destroyAllWindows()

KeyboardInterrupt: 

: 